# Laboratório V · Risco e imposto
*Bitcoin na Veia: do halving ao ETF*, 2ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Para o Capítulo 16, mede o tamanho das quedas do Bitcoin contra as das bolsas e do ouro, a matemática da recuperação, a correlação que sobe na crise, o que aconteceu depois das altas e das quedas fortes (o medo de ficar de fora e o medo de perder tudo em números) e o custo hipotético de um ataque de 51%. Para o Capítulo 17, calcula o imposto da moeda mês a mês, com o degrau da isenção, compara o prejuízo em cada caminho e mostra o imposto do staking numa plataforma brasileira. Para o Capítulo 18, calcula o tamanho da posição pelo risco, quanto bitcoin cabe numa carteira brasileira, o efeito do rebalanceamento e a chance de uma sequência de perdas.

In [ ]:
!pip install -q yfinance

In [2]:
import json
import logging
import os
import urllib.request
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=0: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=0: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
MOEDAS = ["ETH", "XRP", "BNB", "SOL", "DOGE", "ADA", "TRX", "LINK", "XLM", "BCH", "LTC", "AVAX", "DOT",
          "HBAR", "ATOM", "ETC", "XMR", "AAVE", "FIL"]      # a lista do Laboratório IV, sem a UNI (sem dados no Yahoo)
CRIPTO = ["BTC-USD"] + [f"{m}-USD" for m in MOEDAS]
INDICES = {"^GSPC": "S&P 500", "^IXIC": "Nasdaq", "^BVSP": "Ibovespa", "GC=F": "Ouro"}
RAW = "https://raw.githubusercontent.com/traderbrasil/livro-bitcoin-na-veia/main/labs/dados/"
def baixa(nome):
    """Usa a cópia dos dados salva no GitHub do livro (mesmos números do livro); sem ela, baixa tudo de novo."""
    if not os.path.exists(nome):
        try:
            import urllib.request
            urllib.request.urlretrieve(RAW + nome, nome)
        except Exception:
            pass
    return os.path.exists(nome)
CACHE = "dados_lab5.pkl"
if baixa(CACHE):
    CL = pd.read_pickle(CACHE)
else:
    CL = yf.download(CRIPTO + list(INDICES) + ["BRL=X"], start="2014-01-01", end=FIM, auto_adjust=True, progress=False)["Close"]
    pd.to_pickle(CL, CACHE)
btc = CL["BTC-USD"].dropna(); btc = btc[btc > 0]
SERIES = {"Bitcoin": btc} | {nome: CL[t].dropna() for t, nome in INDICES.items()}
print(f"Bitcoin de {btc.index[0]:%d/%m/%Y} a {btc.index[-1]:%d/%m/%Y}; último fechamento US$ {nr(btc.iloc[-1])}")

Bitcoin de 17/09/2014 a 30/09/2026; último fechamento US$ 83.554


## Parte 1: o tamanho das quedas (Capítulo 16)

In [3]:
print("Parte 1a. Queda desde a máxima anterior, de 01/01/2015 a 30/09/2026")
print(f"   {'Ativo':<10} {'Queda máxima':>13} {'Data':>11} {'Tempo >20% abaixo':>18} {'Tempo >50% abaixo':>18} {'Volatilidade':>13}")
for nome, s in SERIES.items():
    s = s.loc["2015-01-01":]; dd = s / s.cummax() - 1
    anual = 365 if nome == "Bitcoin" else 252          # cripto negocia todos os dias
    vol = s.pct_change().std() * np.sqrt(anual)
    print(f"   {nome:<10} {pc(dd.min(), 1):>13} {dd.idxmin():%d/%m/%Y} {pc((dd < -0.2).mean()):>18} {pc((dd < -0.5).mean()):>18} {pc(vol):>13}")

print("\nParte 1b. Cada queda de mais de 20% do Bitcoin, da máxima até o fundo e a volta à máxima")
dd = btc / btc.cummax() - 1
EPIS, atual = [], None
for t, v in dd.items():
    if v == 0:
        if atual and atual["queda"] < -0.2: EPIS.append(atual)
        atual = {"max": t, "fundo": t, "queda": 0.0}
    elif v < atual["queda"]:
        atual["fundo"], atual["queda"] = t, v
if atual and atual["queda"] < -0.2: EPIS.append(atual)
for e in EPIS:
    volta = dd.loc[e["fundo"]:]; volta = volta[volta == 0]
    fim = f"máxima de volta em {volta.index[0]:%d/%m/%Y} ({(volta.index[0] - e['max']).days} dias)" if len(volta) else "ainda abaixo da máxima"
    print(f"   {e['max']:%d/%m/%Y} (US$ {nr(btc[e['max']])}) → {e['fundo']:%d/%m/%Y}: {pc(e['queda'], 1)} | {fim}")
n = lambda x: sum(e["queda"] < x for e in EPIS)
print(f"   Desde 17/09/2014: {len(EPIS)} quedas de mais de 20%, {n(-0.3)} de mais de 30% e {n(-0.5)} de mais de 50%.")

Parte 1a. Queda desde a máxima anterior, de 01/01/2015 a 30/09/2026
   Ativo       Queda máxima        Data  Tempo >20% abaixo  Tempo >50% abaixo  Volatilidade
   Bitcoin           -83,4% 15/12/2018                63%                31%           66%
   S&P 500           -33,9% 23/03/2020                 3%                 0%           18%
   Nasdaq            -36,4% 28/12/2022                10%                 0%           21%
   Ibovespa          -46,8% 23/03/2020                 9%                 0%           23%
   Ouro              -24,9% 16/07/2026                 2%                 0%           17%

Parte 1b. Cada queda de mais de 20% do Bitcoin, da máxima até o fundo e a volta à máxima
   17/09/2014 (US$ 457) → 14/01/2015: -61,1% | máxima de volta em 15/12/2015 (454 dias)
   15/12/2015 (US$ 465) → 15/01/2016: -21,7% | máxima de volta em 26/04/2016 (133 dias)
   16/06/2016 (US$ 766) → 02/08/2016: -28,6% | máxima de volta em 02/12/2016 (169 dias)
   04/01/2017 (US$ 1.155) → 11/

## Parte 2: a matemática da recuperação e os piores dias (Capítulo 16)

In [4]:
print("Parte 2a. Quanto é preciso subir para voltar ao ponto de partida")
for q in [0.10, 0.20, 0.30, 0.50, 0.70, 0.80, 0.90]:
    print(f"   Queda de {pc(q):>4} → alta necessária de {pc(1 / (1 - q) - 1):>5}")
r = btc.pct_change().dropna()
print("\nParte 2b. Os dez piores dias do Bitcoin (fechamento a fechamento, UTC)")
for t, v in r.nsmallest(10).items():
    print(f"   {t:%d/%m/%Y}: {pc(v, 1)}")
ano = r.groupby(r.index.year)
print("\nParte 2c. Dias de queda maior que 10% e dias de variação maior que 5%, por ano")
for a, g in ano:
    print(f"   {a}: {int((g < -0.10).sum()):>2} dias abaixo de −10% | {int((g.abs() > 0.05).sum()):>3} dias com mais de 5% para cima ou para baixo")

Parte 2a. Quanto é preciso subir para voltar ao ponto de partida
   Queda de  10% → alta necessária de   11%
   Queda de  20% → alta necessária de   25%
   Queda de  30% → alta necessária de   43%
   Queda de  50% → alta necessária de  100%
   Queda de  70% → alta necessária de  233%
   Queda de  80% → alta necessária de  400%
   Queda de  90% → alta necessária de  900%

Parte 2b. Os dez piores dias do Bitcoin (fechamento a fechamento, UTC)
   12/03/2020: -37,2%
   14/01/2015: -21,1%
   14/09/2017: -18,7%
   18/08/2015: -18,2%
   16/01/2018: -16,9%
   13/06/2022: -16,0%
   05/02/2018: -16,0%
   13/01/2015: -15,7%
   15/01/2016: -15,3%
   09/11/2022: -14,3%

Parte 2c. Dias de queda maior que 10% e dias de variação maior que 5%, por ano
   2014:  0 dias abaixo de −10% |  13 dias com mais de 5% para cima ou para baixo
   2015:  7 dias abaixo de −10% |  39 dias com mais de 5% para cima ou para baixo
   2016:  2 dias abaixo de −10% |  21 dias com mais de 5% para cima ou para baixo
   2017: 

## Parte 3: a correlação que sobe na crise (Capítulo 16)

In [5]:
J = pd.concat([btc, CL["^IXIC"], CL["GC=F"]], axis=1, keys=["btc", "nasdaq", "ouro"], sort=True).dropna()
R = J.pct_change().dropna()            # só dias de pregão nos EUA; a variação do bitcoin na segunda inclui o fim de semana
print("Parte 3a. Correlação das variações diárias")
for a, b in [("2015", "2019"), ("2020", "2022"), ("2023", "2026")]:
    x = R.loc[a:b]
    print(f"   {a} a {b}: Bitcoin x Nasdaq {nr(x.btc.corr(x.nasdaq), 2)} | Bitcoin x ouro {nr(x.btc.corr(x.ouro), 2)}")
queda_nq = (J.nasdaq / J.nasdaq.cummax() - 1).reindex(R.index) < -0.10
x, s = R.loc["2020":], queda_nq.loc["2020":]
print(f"\nParte 3b. Desde 2020: Bitcoin x Nasdaq {nr(x[~s].btc.corr(x[~s].nasdaq), 2)} com o Nasdaq perto da máxima"
      f" e {nr(x[s].btc.corr(x[s].nasdaq), 2)} com o Nasdaq mais de 10% abaixo dela ({pc(s.mean())} dos dias)")
ruim = x.nasdaq < -0.03
print(f"   Nos {int(ruim.sum())} dias em que o Nasdaq caiu mais de 3%, o bitcoin caiu em {pc((x[ruim].btc < 0).mean())} deles (mediana {pc(x[ruim].btc.median(), 1)})")
fim21, fim22 = "2021-12-31", "2022-12-30"
print(f"   Em 2022: Bitcoin {ps(btc[:fim22].iloc[-1] / btc[:fim21].iloc[-1] - 1)}, Nasdaq {ps(J.nasdaq[:fim22].iloc[-1] / J.nasdaq[:fim21].iloc[-1] - 1)}, ouro {ps(J.ouro[:fim22].iloc[-1] / J.ouro[:fim21].iloc[-1] - 1, 1)}")

ALTS = [f"{m}-USD" for m in MOEDAS]
RA = CL[["BTC-USD"] + ALTS].loc["2021-01-01":].pct_change(fill_method=None)
rb = RA["BTC-USD"]
print("\nParte 3c. As altcoins nos dias de movimento forte do bitcoin, desde 2021")
for nome, m in [("Bitcoin cai mais de 5%", rb < -0.05), ("Bitcoin entre −5% e +5%", rb.abs() <= 0.05), ("Bitcoin sobe mais de 5%", rb > 0.05)]:
    sub = RA.loc[m, ALTS]
    junto = (np.sign(sub).eq(np.sign(rb[m]), axis=0).sum(axis=1) / sub.notna().sum(axis=1)).mean()
    print(f"   {nome:<24} {int(m.sum()):>4} dias | altcoin mediana {ps(sub.median(axis=1).median(), 1):>6} | {pc(junto)} das altcoins na mesma direção")
print("   Na queda, quase tudo cai junto, e mais que o bitcoin. A diversificação entre criptomoedas some quando mais se precisa dela.")

Parte 3a. Correlação das variações diárias
   2015 a 2019: Bitcoin x Nasdaq 0,02 | Bitcoin x ouro 0,06
   2020 a 2022: Bitcoin x Nasdaq 0,44 | Bitcoin x ouro 0,12
   2023 a 2026: Bitcoin x Nasdaq 0,35 | Bitcoin x ouro 0,12

Parte 3b. Desde 2020: Bitcoin x Nasdaq 0,30 com o Nasdaq perto da máxima e 0,52 com o Nasdaq mais de 10% abaixo dela (34% dos dias)
   Nos 50 dias em que o Nasdaq caiu mais de 3%, o bitcoin caiu em 80% deles (mediana -3,2%)
   Em 2022: Bitcoin -64%, Nasdaq -33%, ouro -0,1%

Parte 3c. As altcoins nos dias de movimento forte do bitcoin, desde 2021
   Bitcoin cai mais de 5%     90 dias | altcoin mediana  -8,0% | 96% das altcoins na mesma direção
   Bitcoin entre −5% e +5%  1908 dias | altcoin mediana  -0,1% | 74% das altcoins na mesma direção
   Bitcoin sobe mais de 5%   100 dias | altcoin mediana  +5,4% | 94% das altcoins na mesma direção
   Na queda, quase tudo cai junto, e mais que o bitcoin. A diversificação entre criptomoedas some quando mais se precisa dela.


## Parte 4: o medo de ficar de fora e o medo de perder tudo, em números (Capítulo 16)

In [6]:
r30 = btc / btc.shift(30) - 1
f90 = btc.shift(-90) / btc - 1; f365 = btc.shift(-365) / btc - 1
def inicio(m):
    m = m.fillna(False); return int((m & ~m.shift(1, fill_value=False)).sum())
print("Parte 4. O que veio depois, conforme a variação dos 30 dias anteriores (desde 2014)")
print(f"   {'Situação':<30} {'Dias':>5} {'Episódios':>10} {'90 dias: mediana, % em alta':>29} {'365 dias: mediana, % em alta':>30} {'Pior em 365 dias':>17}")
for nome, m in [("Subiu 50% ou mais em 30 dias", r30 >= 0.5), ("Caiu 30% ou mais em 30 dias", r30 <= -0.3), ("Qualquer dia", r30.notna())]:
    a, b = f90[m].dropna(), f365[m].dropna()
    print(f"   {nome:<30} {int(m.sum()):>5} {inicio(m) if nome != 'Qualquer dia' else '—':>10} {ps(a.median()) + ', ' + pc((a > 0).mean()):>29} {ps(b.median()) + ', ' + pc((b > 0).mean()):>30} {pc(b.min()):>17}")
print("   Comprar depois de uma alta forte não foi ruim nos 90 dias seguintes: o movimento costuma continuar.")
print("   O problema veio depois: em um ano, a mediana ficou muito abaixo da de um dia qualquer, e o pior caso foi −83%.")
print("   Vender depois de uma queda forte também custou caro: um ano depois, a mediana foi de alta.")
print("   Atenção: os dias se sobrepõem. São poucos episódios independentes, e a maior parte vem de 2015 a 2018.")

Parte 4. O que veio depois, conforme a variação dos 30 dias anteriores (desde 2014)
   Situação                        Dias  Episódios   90 dias: mediana, % em alta   365 dias: mediana, % em alta  Pior em 365 dias
   Subiu 50% ou mais em 30 dias     183         31                     +42%, 70%                      +18%, 69%              -83%
   Caiu 30% ou mais em 30 dias      117         22                      +7%, 68%                      +54%, 60%              -68%
   Qualquer dia                    4367          —                      +8%, 59%                      +73%, 73%              -83%
   Comprar depois de uma alta forte não foi ruim nos 90 dias seguintes: o movimento costuma continuar.
   O problema veio depois: em um ano, a mediana ficou muito abaixo da de um dia qualquer, e o pior caso foi −83%.
   Vender depois de uma queda forte também custou caro: um ano depois, a mediana foi de alta.
   Atenção: os dias se sobrepõem. São poucos episódios independentes, e a maior parte

## Parte 5: quanto custaria um ataque de 51% ao Bitcoin (Capítulo 16)

In [7]:
HASH = 945e6           # TH/s em 30/09/2026 (Laboratório I, Parte 7)
TH_MAQ, J_TH, US_MAQ, US_KWH = 200, 17.5, 3_000, 0.05     # hipóteses: máquina moderna de 200 TH/s, 17,5 J/TH, US$ 3 mil, energia a US$ 0,05/kWh
maq = HASH / TH_MAQ
gw = HASH * J_TH / 1e9
print("Parte 5a. Igualar a força de toda a rede (hipóteses ilustrativas; troque pelas suas)")
print(f"   Máquinas: {nr(maq / 1e6, 2)} milhões, a US$ {nr(US_MAQ)} = US$ {nr(maq * US_MAQ / 1e9, 1)} bilhões")
print(f"   Energia: {nr(gw, 1)} GW, US$ {nr(gw * 1e6 * US_KWH / 1e3)} mil por hora, US$ {nr(gw * 1e6 * US_KWH * 24 / 1e6, 1)} milhões por dia")
print(f"   Para comparar: a emissão nova paga 3,125 × 144 = 450 BTC por dia, US$ {nr(450 * btc.iloc[-1] / 1e6, 1)} milhões a US$ {nr(btc.iloc[-1])}")
print("   O gargalo não é só o dinheiro: não existem milhões de máquinas paradas para alugar, e o ataque derrubaria o preço da moeda atacada.")

URL = "https://mempool.space/api/v1/mining/pools/1y"
try:
    POOLS = json.load(urllib.request.urlopen(urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"}), timeout=20))
    origem = "mempool.space, últimos 12 meses, consulta de agora"
except Exception:
    POOLS = {"blockCount": 52357, "pools": [{"name": n, "blockCount": b} for n, b in
             [("Foundry USA", 14684), ("AntPool", 9561), ("F2Pool", 6702), ("ViaBTC", 5025), ("SpiderPool", 4552), ("MARA Pool", 2642)]]}
    origem = "mempool.space, 12 meses até 26/10/2026 (cópia salva)"
tot = POOLS["blockCount"]; P = sorted(POOLS["pools"], key=lambda p: -p["blockCount"])
print(f"\nParte 5b. Blocos por pool ({origem})")
acum = 0
for i, p in enumerate(P[:6], 1):
    acum += p["blockCount"] / tot
    print(f"   {i}. {p['name']:<12} {pc(p['blockCount'] / tot, 1):>6} | acumulado {pc(acum, 1)}")
print("   Dois pools passam de 45% dos blocos. Mas o pool não é dono das máquinas: os mineradores podem trocar de pool em minutos,")
print("   como fizeram com o GHash.io em 2014.")

Parte 5a. Igualar a força de toda a rede (hipóteses ilustrativas; troque pelas suas)
   Máquinas: 4,72 milhões, a US$ 3.000 = US$ 14,2 bilhões
   Energia: 16,5 GW, US$ 827 mil por hora, US$ 19,8 milhões por dia
   Para comparar: a emissão nova paga 3,125 × 144 = 450 BTC por dia, US$ 37,6 milhões a US$ 83.554
   O gargalo não é só o dinheiro: não existem milhões de máquinas paradas para alugar, e o ataque derrubaria o preço da moeda atacada.



Parte 5b. Blocos por pool (mempool.space, últimos 12 meses, consulta de agora)
   1. Foundry USA   28,0% | acumulado 28,0%
   2. AntPool       18,3% | acumulado 46,3%
   3. F2Pool        12,8% | acumulado 59,1%
   4. ViaBTC         9,6% | acumulado 68,7%
   5. SpiderPool     8,7% | acumulado 77,4%
   6. MARA Pool      5,0% | acumulado 82,4%
   Dois pools passam de 45% dos blocos. Mas o pool não é dono das máquinas: os mineradores podem trocar de pool em minutos,
   como fizeram com o GHash.io em 2014.


## Parte 6: o imposto da moeda no Brasil, mês a mês (Capítulo 17)

In [8]:
fx = CL["BRL=X"].ffill()
BRL = (btc * fx.reindex(btc.index).ffill()).dropna()          # bitcoin em reais (fechamento do Yahoo Finance)
ISENCAO = 35_000
FAIXAS = [(5e6, 0.15), (10e6, 0.175), (30e6, 0.20), (float("inf"), 0.225)]   # Lei 8.981/1995, art. 21 (Lei 13.259/2016)
def imposto_ganho(g):
    """Alíquota progressiva por faixa de ganho (cada faixa só sobre a parte que cai nela)."""
    devido, piso = 0.0, 0.0
    for teto, a in FAIXAS:
        if g > piso: devido += (min(g, teto) - piso) * a
        piso = teto
    return devido
# Operações de um investidor fictício (troque pelas suas). Valores em reais; "permuta" = troca por outra cripto (é venda).
OPS = [("2025-01-15", "compra", 20_000), ("2025-03-10", "compra", 20_000), ("2025-04-07", "compra", 20_000),
       ("2025-10-06", "venda", 36_000), ("2025-11-10", "venda", 34_000), ("2025-12-10", "permuta", 30_000),
       ("2026-02-10", "venda", 20_000)]
qtd = custo = 0.0; MESES = {}
print("Parte 6a. Operações, pelo custo médio de aquisição (a permuta por outra cripto conta como venda)")
for d, tipo, valor in OPS:
    p = BRL.loc[d]; q = valor / p
    if tipo == "compra":
        qtd += q; custo += valor
        print(f"   {pd.Timestamp(d):%d/%m/%Y} compra  R$ {nr(valor):>7} a R$ {nr(p):>7} | custo médio R$ {nr(custo / qtd)}")
    else:
        cm = custo / qtd; ganho = valor - q * cm; qtd -= q; custo -= q * cm
        m = MESES.setdefault(pd.Timestamp(d).strftime("%Y-%m"), {"vendas": 0.0, "ganho": 0.0})
        m["vendas"] += valor; m["ganho"] += ganho
        print(f"   {pd.Timestamp(d):%d/%m/%Y} {tipo:<7} R$ {nr(valor):>7} a R$ {nr(p):>7} | ganho R$ {nr(ganho)}")
print("\nParte 6b. O mês fecha assim")
for mes, m in MESES.items():
    venc = (pd.Period(mes, "M") + 1).to_timestamp(how="end").normalize()
    while venc.weekday() >= 5: venc -= pd.Timedelta(days=1)            # último dia útil (sem contar feriados)
    if m["ganho"] <= 0:
        txt = "sem imposto (prejuízo; não abate ganhos de outros meses)"
    elif m["vendas"] <= ISENCAO:
        txt = "isento (vendas até R$ 35 mil)"
    else:
        cod = "1897" if venc >= pd.Timestamp("2026-04-29") else "4600"     # o 1897 foi criado pelo ADE Codar 16/2026 (DOU 29/04/2026)
        txt = f"imposto R$ {nr(imposto_ganho(m['ganho']), 2)} no DARF {cod} até {venc:%d/%m/%Y}"
    print(f"   {mes}: vendas R$ {nr(m['vendas'])} | ganho R$ {nr(m['ganho'])} | {txt}")
print("\nParte 6c. O degrau dos R$ 35 mil (mesmo ganho de 30% sobre o custo)")
for venda in [35_000, 35_001]:
    g = venda - venda / 1.30
    print(f"   Vendas de R$ {nr(venda)} no mês: ganho R$ {nr(g)} → imposto R$ {nr(0 if venda <= ISENCAO else imposto_ganho(g), 2)}")
print("   Um real a mais nas vendas do mês faz o ganho inteiro pagar imposto. A isenção é sobre o total vendido, não sobre o lucro.")
print(f"\nParte 6d. Ganho de R$ 8 milhões: 15% até R$ 5 mi e 17,5% acima = R$ {nr(imposto_ganho(8e6))} ({pc(imposto_ganho(8e6) / 8e6, 2)} efetivo)")

Parte 6a. Operações, pelo custo médio de aquisição (a permuta por outra cripto conta como venda)
   15/01/2025 compra  R$  20.000 a R$ 608.535 | custo médio R$ 608.535
   10/03/2025 compra  R$  20.000 a R$ 454.143 | custo médio R$ 520.123
   07/04/2025 compra  R$  20.000 a R$ 462.737 | custo médio R$ 499.476
   06/10/2025 venda   R$  36.000 a R$ 665.505 | ganho R$ 8.981
   10/11/2025 venda   R$  34.000 a R$ 565.057 | ganho R$ 3.946
   10/12/2025 permuta R$  30.000 a R$ 499.812 | ganho R$ 20
   10/02/2026 venda   R$  20.000 a R$ 357.151 | ganho R$ -7.970

Parte 6b. O mês fecha assim
   2025-10: vendas R$ 36.000 | ganho R$ 8.981 | imposto R$ 1.347,18 no DARF 4600 até 28/11/2025
   2025-11: vendas R$ 34.000 | ganho R$ 3.946 | isento (vendas até R$ 35 mil)
   2025-12: vendas R$ 30.000 | ganho R$ 20 | isento (vendas até R$ 35 mil)
   2026-02: vendas R$ 20.000 | ganho R$ -7.970 | sem imposto (prejuízo; não abate ganhos de outros meses)

Parte 6c. O degrau dos R$ 35 mil (mesmo ganho de 30% so

## Parte 7: o prejuízo em cada caminho (Capítulo 17)

In [9]:
GANHO, PERDA = 50_000, 20_000          # ganho numa posição e perda em outra, no mesmo ano (troque pelos seus)
print(f"Parte 7. Ganho de R$ {nr(GANHO)} numa posição e perda de R$ {nr(PERDA)} em outra, vendidas no mesmo ano, acima da isenção")
print(f"   Moeda no Brasil (ganho de capital): R$ {nr(0.15 * GANHO)}; a perda não abate o ganho")
print(f"   ETF na B3 (renda variável): R$ {nr(0.15 * (GANHO - PERDA))}; a perda abate ganhos em ações e ETFs, inclusive em meses seguintes")
print(f"   Exterior, ETF ou corretora estrangeira (Lei 14.754/2023): R$ {nr(0.15 * (GANHO - PERDA))}, uma vez por ano; a perda abate ganhos no exterior do mesmo ano, e o que sobrar passa para os anos seguintes")
print("   O mesmo resultado antes do imposto pode render imposto diferente conforme o caminho.")

Parte 7. Ganho de R$ 50.000 numa posição e perda de R$ 20.000 em outra, vendidas no mesmo ano, acima da isenção
   Moeda no Brasil (ganho de capital): R$ 7.500; a perda não abate o ganho
   ETF na B3 (renda variável): R$ 4.500; a perda abate ganhos em ações e ETFs, inclusive em meses seguintes
   Exterior, ETF ou corretora estrangeira (Lei 14.754/2023): R$ 4.500, uma vez por ano; a perda abate ganhos no exterior do mesmo ano, e o que sobrar passa para os anos seguintes
   O mesmo resultado antes do imposto pode render imposto diferente conforme o caminho.


## Parte 8: staking numa plataforma brasileira (Capítulo 17)

In [10]:
REGRESSIVA = [(180, 0.225), (360, 0.20), (720, 0.175), (10**9, 0.15)]   # Lei 11.033/2004, art. 1º
RECOMPENSA = 1_000                     # R$ por mês, pelo valor de mercado no dia do recebimento (troque pelo seu)
print("Parte 8. Solução de Consulta Cosit 184/2024: a remuneração da cessão temporária de cripto a um custodiante no Brasil")
print("   é rendimento, com imposto retido na fonte pela tabela da renda fixa, mesmo se paga em cripto")
for dias, a in REGRESSIVA:
    faixa = f"até {dias} dias" if dias < 10**9 else "mais de 720 dias"
    print(f"   {faixa:<17}: {pc(a, 1):>5} → R$ {nr(RECOMPENSA * a, 2)} retidos sobre R$ {nr(RECOMPENSA)}")
print("   A moeda recebida entra com custo igual ao valor tributado. Quando for vendida, o ganho conta a partir dali.")
print("   Staking em carteira própria ou em DeFi não tem regra específica da Receita: confira com um contador.")

Parte 8. Solução de Consulta Cosit 184/2024: a remuneração da cessão temporária de cripto a um custodiante no Brasil
   é rendimento, com imposto retido na fonte pela tabela da renda fixa, mesmo se paga em cripto
   até 180 dias     : 22,5% → R$ 225,00 retidos sobre R$ 1.000
   até 360 dias     : 20,0% → R$ 200,00 retidos sobre R$ 1.000
   até 720 dias     : 17,5% → R$ 175,00 retidos sobre R$ 1.000
   mais de 720 dias : 15,0% → R$ 150,00 retidos sobre R$ 1.000
   A moeda recebida entra com custo igual ao valor tributado. Quando for vendida, o ganho conta a partir dali.
   Staking em carteira própria ou em DeFi não tem regra específica da Receita: confira com um contador.


## Parte 9: o tamanho da posição pelo risco (Capítulo 18)

In [11]:
CONTA, RISCO = 100_000, 0.01          # patrimônio para operações e risco por operação (troque pelos seus)
ATR = {2017: 0.056, 2021: 0.057, 2023: 0.029, 2024: 0.038, 2025: 0.031, 2026: 0.030}   # ATR mediano por ano (Laboratório IV, Parte 6)
print(f"Parte 9. Conta de R$ {nr(CONTA)}, risco de {pc(RISCO)} por operação (R$ {nr(CONTA * RISCO)}), stop a 2 ATR")
for ano, a in ATR.items():
    stop = 2 * a; pos = CONTA * RISCO / stop
    print(f"   {ano}: stop de {pc(stop, 1):>5} → posição de R$ {nr(pos):>7} ({pc(pos / CONTA)} da conta)")
print("   O mesmo risco em reais pede uma posição duas vezes menor quando o bitcoin fica duas vezes mais volátil.")
print("\n   O mesmo cálculo com stop de 30% (uma queda comum do bitcoin) e risco de 1%: posição de R$ " + nr(CONTA * RISCO / 0.30)
      + f" ({pc(RISCO / 0.30, 1)} da conta)")

Parte 9. Conta de R$ 100.000, risco de 1% por operação (R$ 1.000), stop a 2 ATR
   2017: stop de 11,2% → posição de R$   8.929 (9% da conta)
   2021: stop de 11,4% → posição de R$   8.772 (9% da conta)
   2023: stop de  5,8% → posição de R$  17.241 (17% da conta)
   2024: stop de  7,6% → posição de R$  13.158 (13% da conta)
   2025: stop de  6,2% → posição de R$  16.129 (16% da conta)
   2026: stop de  6,0% → posição de R$  16.667 (17% da conta)
   O mesmo risco em reais pede uma posição duas vezes menor quando o bitcoin fica duas vezes mais volátil.

   O mesmo cálculo com stop de 30% (uma queda comum do bitcoin) e risco de 1%: posição de R$ 3.333 (3,3% da conta)


## Parte 10: quanto bitcoin cabe na carteira (Capítulo 18)

In [12]:
import json, urllib.request
CDICACHE = "cdi_lab5.pkl"
if baixa(CDICACHE):
    CDI = pd.read_pickle(CDICACHE)
else:   # Banco Central, SGS série 12: CDI diário, em % ao dia
    partes = []
    for a, b in [("01/01/2015", "31/12/2024"), ("01/01/2025", "30/09/2026")]:
        url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados?formato=json&dataInicial={a}&dataFinal={b}"
        partes.append(pd.Series({pd.to_datetime(x["data"], dayfirst=True): float(x["valor"]) / 100
                                 for x in json.load(urllib.request.urlopen(url, timeout=60))}))
    CDI = pd.concat(partes).sort_index(); pd.to_pickle(CDI, CDICACHE)
BTC_BRL = (btc * CL["BRL=X"].ffill().reindex(btc.index).ffill()).dropna()
MES = pd.DataFrame({"cdi": (1 + CDI).cumprod(), "ibov": CL["^BVSP"].dropna(), "btc": BTC_BRL}).ffill()
MES = MES.loc["2015-01-01":"2026-09-30"].resample("ME").last().dropna()
RM = MES.pct_change().dropna()
BASE = (0.70, 0.30)          # carteira base: 70% CDI e 30% Ibovespa (troque pela sua)
def carteira(w_btc, R, rebal="mensal", banda=None):
    alvo = np.array([BASE[0] * (1 - w_btc), BASE[1] * (1 - w_btc), w_btc]); pos = alvo.copy(); v, peso = [], []
    for t, r in R.iterrows():
        pos = pos * (1 + r.values); tot = pos.sum(); v.append(tot); peso.append(pos[2] / tot)
        if rebal == "mensal" or (rebal == "anual" and t.month == 12) or (banda and abs(pos[2] / tot - w_btc) > banda * w_btc):
            pos = tot * alvo
    v = pd.Series(v, R.index); r_ = v.pct_change().fillna(v.iloc[0] - 1)
    return v.iloc[-1] ** (12 / len(v)) - 1, r_.std() * np.sqrt(12), (v / v.cummax() - 1).min(), max(peso)
print("Parte 10a. Carteira de 70% CDI e 30% Ibovespa, com bitcoin em reais, rebalanceada todo mês (retorno ao ano | volatilidade | maior queda nos fechamentos mensais)")
for inicio, nome in [("2015-02-01", "de fev/2015"), ("2018-01-01", "de jan/2018 (topo)"), ("2021-11-01", "de nov/2021 (topo)")]:
    R = RM.loc[inicio:]
    linha = " | ".join(f"{pc(w)}: {pc(c, 1)}, {pc(s, 1)}, {pc(d, 1)}" for w in [0, 0.02, 0.05, 0.10] for c, s, d, _ in [carteira(w, R)])
    print(f"   {nome:<20} {linha}")
cov = RM.cov().values * 12
print("\nParte 10b. Quanto do risco da carteira vem do bitcoin")
for w in [0.01, 0.02, 0.05, 0.10]:
    pesos = np.array([BASE[0] * (1 - w), BASE[1] * (1 - w), w]); var = pesos @ cov @ pesos
    print(f"   {pc(w):>3} de bitcoin → {pc(pesos[2] * (cov @ pesos)[2] / var, 1):>5} do risco")
print("   Com 5% do dinheiro, o bitcoin responde por um quarto do risco; com 10%, por mais da metade.")

Parte 10a. Carteira de 70% CDI e 30% Ibovespa, com bitcoin em reais, rebalanceada todo mês (retorno ao ano | volatilidade | maior queda nos fechamentos mensais)
   de fev/2015          0%: 11,3%, 6,5%, -11,1% | 2%: 12,8%, 6,5%, -11,0% | 5%: 15,2%, 7,0%, -11,1% | 10%: 19,0%, 9,1%, -11,3%
   de jan/2018 (topo)   0%: 10,2%, 6,5%, -11,1% | 2%: 11,0%, 6,5%, -11,0% | 5%: 12,2%, 7,0%, -11,1% | 10%: 14,1%, 8,9%, -11,3%
   de nov/2021 (topo)   0%: 13,1%, 5,1%, -3,5% | 2%: 13,2%, 5,1%, -4,7% | 5%: 13,4%, 5,5%, -6,5% | 10%: 13,6%, 7,0%, -9,4%

Parte 10b. Quanto do risco da carteira vem do bitcoin
    1% de bitcoin →  1,0% do risco
    2% de bitcoin →  4,4% do risco
    5% de bitcoin → 24,7% do risco
   10% de bitcoin → 60,1% do risco
   Com 5% do dinheiro, o bitcoin responde por um quarto do risco; com 10%, por mais da metade.


## Parte 11: rebalancear ou deixar correr (Capítulo 18)

In [13]:
print("Parte 11. Alvo de 5% de bitcoin, de fev/2015 a set/2026 (retorno ao ano | volatilidade | maior queda nos fechamentos mensais | maior peso do bitcoin)")
for nome, kw in [("todo mês", dict(rebal="mensal")), ("uma vez por ano", dict(rebal="anual")),
                 ("quando o peso sai de 2,5% a 7,5%", dict(rebal=None, banda=0.5)), ("nunca", dict(rebal=None))]:
    c, s, d, m = carteira(0.05, RM, **kw)
    print(f"   {nome:<34} {pc(c, 1):>6} | {pc(s, 1):>6} | {pc(d, 1):>7} | {pc(m, 1):>6}")
print("   Sem rebalancear, os 5% viraram quase a carteira inteira: o retorno passado foi enorme, e o risco também.")
print("   Rebalancear é vender um pouco do que subiu e comprar do que caiu: mantém o risco no tamanho escolhido.")

Parte 11. Alvo de 5% de bitcoin, de fev/2015 a set/2026 (retorno ao ano | volatilidade | maior queda nos fechamentos mensais | maior peso do bitcoin)
   todo mês                            15,2% |   7,0% |  -11,1% |   8,4%
   uma vez por ano                     19,6% |   9,8% |  -10,5% |  40,6%
   quando o peso sai de 2,5% a 7,5%    15,4% |   7,2% |  -11,5% |  10,4%


   nunca                               37,6% |  49,1% |  -70,1% |  95,8%
   Sem rebalancear, os 5% viraram quase a carteira inteira: o retorno passado foi enorme, e o risco também.
   Rebalancear é vender um pouco do que subiu e comprar do que caiu: mantém o risco no tamanho escolhido.


## Parte 12: sobreviver a uma sequência de perdas (Capítulo 18)

In [14]:
rng = np.random.default_rng(42)
N_OPS, SIM = 100, 20_000
print(f"Parte 12a. Chance de ter pelo menos uma sequência de perdas seguidas em {N_OPS} operações ({nr(SIM)} simulações)")
for acerto in [0.40, 0.50]:
    ganhos = rng.random((SIM, N_OPS)) < acerto
    maior = np.zeros(SIM, dtype=int); atual = np.zeros(SIM, dtype=int)
    for j in range(N_OPS):
        atual = np.where(ganhos[:, j], 0, atual + 1); maior = np.maximum(maior, atual)
    print(f"   Acerto de {pc(acerto)}: 5 perdas seguidas {pc((maior >= 5).mean())}, 8 seguidas {pc((maior >= 8).mean())}, 10 seguidas {pc((maior >= 10).mean())}")
print("\nParte 12b. O que sobra da conta depois de 10 perdas seguidas")
for r in [0.01, 0.02, 0.05, 0.10, 0.20]:
    sobra = (1 - r) ** 10
    print(f"   Risco de {pc(r):>3} por operação → sobra {pc(sobra, 1):>5}; para voltar, precisa subir {pc(1 / sobra - 1):>5}")
print("   Com 1% por operação, dez erros seguidos custam 9,6%. Com 10%, custam dois terços da conta.")

Parte 12a. Chance de ter pelo menos uma sequência de perdas seguidas em 100 operações (20.000 simulações)


   Acerto de 40%: 5 perdas seguidas 98%, 8 seguidas 50%, 10 seguidas 21%
   Acerto de 50%: 5 perdas seguidas 81%, 8 seguidas 17%, 10 seguidas 4%

Parte 12b. O que sobra da conta depois de 10 perdas seguidas
   Risco de  1% por operação → sobra 90,4%; para voltar, precisa subir   11%
   Risco de  2% por operação → sobra 81,7%; para voltar, precisa subir   22%
   Risco de  5% por operação → sobra 59,9%; para voltar, precisa subir   67%
   Risco de 10% por operação → sobra 34,9%; para voltar, precisa subir  187%
   Risco de 20% por operação → sobra 10,7%; para voltar, precisa subir  831%
   Com 1% por operação, dez erros seguidos custam 9,6%. Com 10%, custam dois terços da conta.


## Para ir além
1. Na Parte 1, meça as quedas do bitcoin em reais. O dólar ajudou ou atrapalhou quem é brasileiro?
2. Na Parte 1, troque o ouro pela prata (SI=F). Qual dos dois se parece mais com o bitcoin?
3. Na Parte 3, use a correlação de 90 dias em janela móvel e marque os meses de 2022. Quando ela subiu?
4. Na Parte 3c, separe as altcoins grandes (ETH, XRP, BNB, SOL) das pequenas. Quem cai mais nos dias ruins?
5. Na Parte 4, troque 30 dias por 7 e o limite de 50% por 20%. A conclusão muda?
6. Na Parte 4, conte só um dia por episódio (o primeiro). A mediana de 365 dias muda muito?
7. Na Parte 5, refaça a conta para uma moeda pequena de prova de trabalho, como o Bitcoin Cash. Quanto custaria uma hora de ataque?
8. Na Parte 6, troque as suas operações reais pelas do exemplo. Algum mês passou dos R$ 35 mil sem você perceber?
9. Na Parte 6, deixe R$ 1.000 da venda de outubro para dezembro. Quanto imposto some?
10. Na Parte 7, troque a perda por um ganho menor. Em qual caminho a diferença some?
11. Na Parte 9, troque o risco de 1% por 0,5% e por 2%. Qual posição você aguentaria ver cair 30%?
12. Na Parte 10, troque a carteira base pela sua. A partir de que peso o bitcoin passa a ser o maior risco?
13. Na Parte 11, teste bandas de 25% e de 100% em torno do alvo. Quantas vezes você teria rebalanceado?
14. Na Parte 12, use a sua taxa de acerto real. Qual sequência de perdas você deve esperar em um ano?

**Governança:** os resultados valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo.

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros.

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido.

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.